# Diffusion coefficients for hybrid quantum walks

This notebook extracts numerical diffusion coefficients from simulation output and evaluates the analytical expressions used in the manuscript. Run it from the repository root.


## Numerical extraction from the MSD


In [ ]:
"""
MSD analysis: slope profile and extraction of the diffusion coefficient D

The procedure follows the original slope-profile analysis:
1. Calculate the blockwise log-log slope γ(t).
2. Fit ⟨x²⟩ = 2D·t within intervals satisfying |γ - 1| < ε.
3. Return D = NaN when fewer than min_blocks diffusive blocks are found.
"""

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


def slope_profile_with_D(times, msd, block=200, epsilon=0.05, min_blocks=3):
    """
    Calculate blockwise log-log slopes and extract D from diffusive intervals.

    Parameters:
        times: Time array (0, 1, 2, ...)
        msd: MSD array
        block: Number of time steps per block
        epsilon: A block is diffusive when |γ - 1| < epsilon
        min_blocks: Return D = NaN if fewer blocks are diffusive

    Returns:
        dict:
            t_centers: Times at block centers
            slopes: Blockwise slope γ
            is_diffusive: Boolean mask for diffusive blocks
            D_values: D for diffusive blocks and NaN otherwise
            D_mean: Mean D over diffusive blocks, or NaN if unavailable
            D_std: Standard deviation of D, or NaN if unavailable
            diffusive_range: Time range (t_start, t_end) of diffusive blocks
    """
    mask = times > 0
    t_pos = times[mask].astype(float)
    msd_pos = msd[mask]
    log_t = np.log(t_pos)
    log_msd = np.log(msd_pos)

    n_blocks = (len(log_t) - block) // block + 1

    t_centers = np.zeros(n_blocks)
    slopes = np.zeros(n_blocks)
    D_values = np.full(n_blocks, np.nan)
    is_diffusive = np.zeros(n_blocks, dtype=bool)

    for i in range(n_blocks):
        i0 = i * block
        i1 = i0 + block
        if i1 > len(log_t):
            break

        # Log-log slope
        gamma, _ = np.polyfit(log_t[i0:i1], log_msd[i0:i1], 1)
        t_centers[i] = t_pos[i0 + block // 2]
        slopes[i] = gamma

        # Diffusive criterion: |γ - 1| < ε
        if abs(gamma - 1.0) < epsilon:
            is_diffusive[i] = True

            # Linear fit within this block: ⟨x²⟩ = 2D·t + c
            t_block = t_pos[i0:i1]
            msd_block = msd_pos[i0:i1]
            try:
                coeffs = np.polyfit(t_block, msd_block, 1)
                D_values[i] = coeffs[0] / 2.0  # slope = 2D → D = slope/2
            except Exception:
                D_values[i] = np.nan

    # Statistics over diffusive blocks after the min_blocks filter
    diff_mask = is_diffusive & ~np.isnan(D_values)
    if diff_mask.sum() >= min_blocks:
        D_mean = np.nanmean(D_values[diff_mask])
        D_std = np.nanstd(D_values[diff_mask])
        diff_times = t_centers[diff_mask]
        diffusive_range = (diff_times[0], diff_times[-1])
    else:
        D_mean = np.nan
        D_std = np.nan
        diffusive_range = (np.nan, np.nan)

    return {
        't_centers': t_centers,
        'slopes': slopes,
        'is_diffusive': is_diffusive,
        'D_values': D_values,
        'D_mean': D_mean,
        'D_std': D_std,
        'diffusive_range': diffusive_range,
        'n_diffusive_blocks': int(diff_mask.sum()),
        'min_blocks': min_blocks,
        'epsilon': epsilon,
        'block': block,
    }


def analyze_diffusion_profile(filepath, block=200, epsilon=0.05, min_blocks=3):
    """
    Extract the slope profile and D from one .npz file.

    Returns:
        info: File metadata
        profile: Output of slope_profile_with_D
    """
    data = np.load(filepath, allow_pickle=True)
    msd = data['msd_list']
    times = np.arange(len(msd))

    info = {
        'filename': Path(filepath).name,
        'walk_type': str(data['walk_type']),
        'graph_type': str(data['graph_type']),
        'N': int(data['num_vertices']),
        'n_qw': int(data['n_qw']),
        'n_rw': int(data['n_rw']),
        'cycle_length': int(data['n_qw']) + int(data['n_rw']),
        'alpha': float(data['alpha']),
        'time_length': int(data['time_length']),
    }

    profile = slope_profile_with_D(times, msd, block, epsilon, min_blocks)

    return info, profile


def analyze_sweep_diffusion(result_dir, block=200, epsilon=0.05, min_blocks=3,
                            sort_by='D_mean', ascending=True):
    """
    Analyze slopes and D for every .npz file in a directory.

    Parameters:
        result_dir: Directory containing .npz files
        block: Number of time steps per block
        epsilon: Diffusive-block tolerance
        min_blocks: Minimum number of diffusive blocks
        sort_by: Key used for sorting (for example, 'D_mean', 'alpha', or 'n_qw')
        ascending: True for ascending order and False for descending order

    Returns:
        records: Sorted list of per-file summary dictionaries
        profiles: List of (info, profile) pairs in the same order as records
    """
    result_dir = Path(result_dir)
    files = sorted(result_dir.glob('*.npz'))

    if not files:
        print(f"No .npz files in {result_dir}")
        return [], []

    records = []
    profiles = []

    for f in files:
        info, profile = analyze_diffusion_profile(f, block, epsilon, min_blocks)
        profiles.append((info, profile))

        records.append({
            **info,
            'D_mean': profile['D_mean'],
            'D_std': profile['D_std'],
            'diffusive_t_start': profile['diffusive_range'][0],
            'diffusive_t_end': profile['diffusive_range'][1],
            'n_diffusive_blocks': profile['n_diffusive_blocks'],
        })

    # Sort with NaN values last
    def sort_key(r):
        val = r.get(sort_by, np.nan)
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return (1, 0)  # Place NaN last
        return (0, val if ascending else -val)

    sorted_indices = sorted(range(len(records)), key=lambda i: sort_key(records[i]))
    records = [records[i] for i in sorted_indices]
    profiles = [profiles[i] for i in sorted_indices]

    print(f"Analyzed {len(records)} files (block={block}, ε={epsilon}, min_blocks={min_blocks})")
    print(f"Sorted by '{sort_by}' ({'asc' if ascending else 'desc'})")
    return records, profiles


# ============================================================
# Visualization
# ============================================================

def plot_slope_and_D(info, profile, figsize=(14, 5)):
    """
    Two panels: γ(t) on the left and D(t) over diffusive blocks on the right.
    """
    tc = profile['t_centers']
    slopes = profile['slopes']
    D_values = profile['D_values']
    is_diff = profile['is_diffusive']
    eps = profile['epsilon']
    block = profile['block']
    min_blk = profile['min_blocks']

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # Left: slope profile
    ax = axes[0]
    ax.plot(tc, slopes, 'b-o', markersize=2, alpha=0.7, label='γ(t)')

    if is_diff.any():
        ax.scatter(tc[is_diff], slopes[is_diff], c='red', s=15, zorder=5,
                   label=f'|γ-1| < {eps}')

    ax.axhline(1.0, color='k', linestyle='--', alpha=0.4, label='γ=1 (diffusive)')
    ax.axhspan(1.0 - eps, 1.0 + eps, color='red', alpha=0.08)
    ax.axhline(2.0, color='g', linestyle='--', alpha=0.3, label='γ=2 (ballistic)')

    wt = info['walk_type']
    if wt == 'QRW_M':
        param_str = f"n_qw={info['n_qw']}, n_rw={info['n_rw']}"
    elif wt == 'QRW_A':
        param_str = f"α={info['alpha']}"
    else:
        param_str = ''

    ax.set_xlabel('t')
    ax.set_ylabel(f'γ (block={block})')
    ax.set_title(f'{wt} {param_str} | {info["graph_type"]} N={info["N"]}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Right: D(t)
    ax = axes[1]
    diff_mask = is_diff & ~np.isnan(D_values)
    n_diff = diff_mask.sum()

    if n_diff >= min_blk:
        ax.plot(tc[diff_mask], D_values[diff_mask], 'r-o', markersize=3,
                alpha=0.7, label='D(t)')
        ax.axhline(profile['D_mean'], color='k', linestyle='--', alpha=0.5,
                   label=f'⟨D⟩ = {profile["D_mean"]:.4f} ± {profile["D_std"]:.4f}')
        ax.fill_between(
            [tc[diff_mask][0], tc[diff_mask][-1]],
            profile['D_mean'] - profile['D_std'],
            profile['D_mean'] + profile['D_std'],
            color='gray', alpha=0.15
        )
        ax.set_title(f'D = {profile["D_mean"]:.4f} ± {profile["D_std"]:.4f}  ({n_diff} blocks)')
    else:
        msg = f'Diffusive blocks: {n_diff} < {min_blk} (min)'
        ax.text(0.5, 0.5, msg,
                transform=ax.transAxes, ha='center', fontsize=11, color='gray')
        ax.set_title(f'D: N/A ({n_diff} blocks < {min_blk})')

    ax.set_xlabel('t')
    ax.set_ylabel('D')
    if n_diff >= min_blk:
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return fig


def plot_D_vs_param(records, param_key='n', title=None, figsize=(8, 5)):
    """Plot D against a parameter after sorting by that parameter."""
    if not records:
        print("No diffusion records to plot.")
        return None

    import pandas as pd
    df = pd.DataFrame(records)
    df = df.dropna(subset=['D_mean']).sort_values(param_key).reset_index(drop=True)

    if df.empty:
        print("No diffusive data to plot.")
        return None

    fig, ax = plt.subplots(figsize=figsize)
    ax.errorbar(df[param_key], df['D_mean'], yerr=df['D_std'],
                marker='o', capsize=3, linewidth=1)
    ax.set_xlabel(param_key)
    ax.set_ylabel('D (diffusion coefficient)')
    ax.set_title(title or f'D vs {param_key}')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return df

## QRW-M: numerical diffusion coefficients


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()  # Run this notebook from the repository root

# Use the numerical procedure described in the Supplemental Material
records, profiles = analyze_sweep_diffusion(
    PROJECT_DIR / 'results/qrw_m/',
    block=200, epsilon=0.05, min_blocks=3,
    sort_by='n_qw', ascending=True
)

# 1. Plot the slope profile and D for each file
for info, profile in profiles:
    plot_slope_and_D(info, profile)

# 2. Plot D against n_qw
df = plot_D_vs_param(records, param_key='n_qw', title='QRW-M: D vs n_qw')
if df is not None:
    print(df[['n_qw', 'n_rw', 'cycle_length', 'D_mean', 'D_std', 'diffusive_t_start', 'diffusive_t_end', 'n_diffusive_blocks']].to_string())
else:
    # Show all records even when no diffusive interval is detected
    if records:
        import pandas as pd
        df_all = pd.DataFrame(records)
        print(df_all[['n_qw', 'n_rw', 'cycle_length', 'D_mean', 'n_diffusive_blocks']].to_string())
    else:
        print("No QRW-M result files were found.")

## QRW-M: exact Laurent-polynomial recurrence


In [ ]:
from fractions import Fraction
from collections import defaultdict


def laurent_mul(a, b):
    result = defaultdict(lambda: Fraction(0))
    for pa, ca in a.items():
        for pb, cb in b.items():
            result[pa + pb] += ca * cb
    return {p: c for p, c in result.items() if c != 0}


def laurent_add(a, b, scale_b=Fraction(1)):
    result = defaultdict(lambda: Fraction(0))
    for p, c in a.items():
        result[p] += c
    for p, c in b.items():
        result[p] += scale_b * c
    return {p: c for p, c in result.items() if c != 0}


def v2_fourier0(n):
    mult = {-1: Fraction(-2), 0: Fraction(-4), 1: Fraction(-2)}
    T_prev2 = {0: Fraction(2)}
    T_prev1 = {-1: Fraction(-2), 0: Fraction(-4), 1: Fraction(-2)}
    m = n + 1
    if m == 0:
        T = T_prev2
    elif m == 1:
        T = T_prev1
    else:
        for j in range(2, m + 1):
            T = laurent_add(laurent_mul(mult, T_prev1), T_prev2, Fraction(-16))
            T_prev2 = T_prev1
            T_prev1 = T
    four_n = Fraction(4) ** n
    Tz2 = {p + 2: c / four_n for p, c in T.items()}
    K = -min(Tz2.keys()) + 5
    a = [Fraction(0)] * (K + 1)
    a[0] = Fraction(1)
    if K >= 1:
        a[1] = Fraction(-6)
    for k in range(2, K + 1):
        a[k] = -6 * a[k-1] - a[k-2]
    d = defaultdict(lambda: Fraction(0))
    for i in range(K + 1):
        for j in range(K + 1):
            if i + j <= K:
                d[i + j] += a[i] * a[j]
    ce_part = Fraction(0)
    for p, c in Tz2.items():
        if -p >= 0 and -p in d:
            ce_part += c * d[-p]
    rational_part = Fraction(-(n + 1)**2, 2)
    return rational_part, ce_part, rational_part + ce_part


if __name__ == "__main__":
    print("  n      total/n^2       CE/n^2      (total+n^2/2)/n")
    print("-" * 60)
    for n in [5, 10, 20, 30, 50, 75, 100, 150, 200]:
        rat, ce, total = v2_fourier0(n)
        t = float(total)
        c = float(ce)
        print(f"{n:>4}   {t/n**2:>12.8f}   {c/n**2:>12.8f}   {(t + n**2/2)/n:>12.8f}")


def exact_D_M(n_qw):
    """Return the exact QRW-M diffusion coefficient for n_qw quantum steps."""
    _, _, r2_zero = v2_fourier0(n_qw)
    return -r2_zero / Fraction(n_qw + 1)


print("\nExact QRW-M diffusion coefficients")
print("n_qw        D_M")
print("-" * 24)
for n_qw in range(20):
    value = exact_D_M(n_qw)
    print(f"{n_qw:>4}  {str(value):>9}  ({float(value):.8f})")


## QRW-A: numerical diffusion coefficients


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()  # Run this notebook from the repository root

# Analyze all files with the numerical procedure from the Supplemental Material
records_a, profiles_a = analyze_sweep_diffusion(
    PROJECT_DIR / 'results/qrw_a/',
    block=200,
    epsilon=0.05,
    min_blocks=3
)

# 1. Plot the slope profile and D for each file
for info, profile in profiles_a:
    plot_slope_and_D(info, profile)

# 2. Plot D against alpha
df_a = plot_D_vs_param(records_a, param_key='alpha', title='QRW_A: D vs α')
if df_a is not None:
    print(df_a[['alpha', 'D_mean', 'D_std', 'diffusive_t_start', 'diffusive_t_end', 'n_diffusive_blocks']].to_string())

## QRW-A: transfer-operator truncations


In [ ]:
def D_A_L1(alpha):
    """QRW-A diffusion coefficient from the L=1 transfer-operator truncation."""
    return (2 + alpha**3 + alpha**4) / (2 * (2 - alpha**3 - alpha**4))


def D_A_L2(alpha):
    """QRW-A diffusion coefficient from the L=2 transfer-operator truncation."""
    numerator = 4 + 4*alpha + 3*alpha**2 + 3*alpha**3 + 4*alpha**4 + 4*alpha**5 + 2*alpha**6
    denominator = 2 * (4 + 4*alpha + 3*alpha**2 - alpha**3 - 4*alpha**4 - 4*alpha**5 - 2*alpha**6)
    return numerator / denominator


alpha = 0.95
print(f"alpha={alpha}: D_A(L=1)={D_A_L1(alpha):.8f}, D_A(L=2)={D_A_L2(alpha):.8f}")
